In [1]:
# 1. Import libraries
from pathlib import Path

import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import rasterio
import seaborn as sns
from rasterstats import zonal_stats
import matplotlib.dates as mdates


In [128]:
# 2. Set fire inputs and paths
PROJ_DIR = Path('/Users/dalo2903/Library/CloudStorage/OneDrive-UCB-O365/Documents/dev/intensity_x_severity/')
VIIRS_GPKG = Path(PROJ_DIR,"westUS_FIRED_daily_viirs_gridstats_2012-2025_final.gpkg")
CBI_DIR = Path(PROJ_DIR)
OUTPUT_DIR = Path(PROJ_DIR,"single_fire_ouputs")

FIRE_NAME = "Holiday Farm" #"DoubleCreek" #"Pearl Hill" #"Archie Creek" #"Beachie Creek" #"Pine Gulch" #"Bootleg" #"Carlton Complex" #"Riverside" #"Canyon Creek Complex"
FIRE_ID = 40246 #37673 #199886 #42103 #48602 #44011 #199758 #38454 #40024 #
FIRE_DATE = "2020-01-01"# "2014-01-01" #"2020-01-01" #"2015-01-01"  # YYYY-MM-DD

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

In [129]:
# 3. Helper functions
def duration_category(duration_days):
    """Classify fires into short vs long duration groups."""
    if pd.isna(duration_days):
        return "unknown"
    if duration_days < 1:
        return "0-1 days"
    elif duration_days <= 2:
        return "1-2 days"
    else:
        return "3+ days"


def get_cbi_path(cbi_dir, fire_date):
    year = pd.to_datetime(fire_date).year
    cbi_path = Path(cbi_dir) / f"{year}_pnw_firedv2_bccbi_epsg5070.tif"
    if not cbi_path.exists():
        raise FileNotFoundError(f"No CBI raster for year {year}: {cbi_path}")
    return cbi_path


In [130]:
# 4. Load VIIRS and isolate one fire
viirs = gpd.read_file(VIIRS_GPKG)
viirs_singlefire = viirs.loc[viirs["id"] == FIRE_ID].copy()

if viirs_singlefire.empty:
    raise ValueError(f"No VIIRS records found for id={FIRE_ID}")

summary = pd.DataFrame([
    {
        "fire_name": FIRE_NAME,
        "fire_id": FIRE_ID,
        "fire_date": FIRE_DATE,
        "n_cells": len(viirs_singlefire),
        "frp_norm_csum_mean": viirs_singlefire["frp_norm_csum"].mean(),
        "frp_norm_csum_median": viirs_singlefire["frp_norm_csum"].median(),
        "frp_norm_csum_min": viirs_singlefire["frp_norm_csum"].min(),
        "frp_norm_csum_max": viirs_singlefire["frp_norm_csum"].max(),
        "obs_duration_mean": viirs_singlefire["obs_duration"].mean(),
        "obs_duration_max": viirs_singlefire["obs_duration"].max(),
    }
])
summary

,fire_name,fire_id,fire_date,n_cells,frp_norm_csum_mean,frp_norm_csum_median,frp_norm_csum_min,frp_norm_csum_max,obs_duration_mean,obs_duration_max
0,Holiday Farm,40246,2020-01-01,5998,956.973958,567.967831,0.939542,9020.908005,1.896799,28


In [131]:
# 5. Join per-cell CBI stats using zonal statistics
cbi_path = get_cbi_path(CBI_DIR, FIRE_DATE)
with rasterio.open(cbi_path) as src:
    cbi_array = src.read(1, masked=True).filled(np.nan)
    cbi_transform = src.transform

downsample_stats = zonal_stats(
    viirs_singlefire,
    cbi_array,
    affine=cbi_transform,
    stats=["mean", "median", "min", "max", "percentile_95", "std"],
    nodata=np.nan,
    all_touched=True,
)

viirs_singlefire["cbi_mean"] = [s["mean"] for s in downsample_stats]
viirs_singlefire["cbi_stdev"] = [s["std"] for s in downsample_stats]
viirs_singlefire["cbi_median"] = [s["median"] for s in downsample_stats]
viirs_singlefire["cbi_min"] = [s["min"] for s in downsample_stats]
viirs_singlefire["cbi_max"] = [s["max"] for s in downsample_stats]
viirs_singlefire["cbi_95"] = [s["percentile_95"] for s in downsample_stats]

# modify variables
viirs_singlefire["log_cfrp"] = np.log(viirs_singlefire["frp_norm_csum"] + 1)
viirs_singlefire["duration_group"] = viirs_singlefire["obs_duration"].apply(duration_category)
viirs_singlefire["first_obs_date"] = pd.to_datetime(viirs_singlefire["first_obs_date"])
viirs_singlefire["first_obs_ordinal"] = viirs_singlefire["first_obs_date"].map(pd.Timestamp.toordinal)


bins = [0, 500, 1000, 2000, viirs_singlefire["frp_norm_csum"].max()]
labels = ["<500", "500-1000", "1000-2000", "2000+"]
viirs_singlefire["intensity_class"] = pd.cut(
    viirs_singlefire["frp_norm_csum"],
    bins=bins,
    labels=labels,
    include_lowest=True)

viirs_singlefire.head()

,grid_index,id,assigned_did,overlap_area,overlap_fraction,assignment_method,afd_count,frp_norm_csum,frp_norm_max,frp_norm_min,...,cbi_mean,cbi_stdev,cbi_median,cbi_min,cbi_max,cbi_95,log_cfrp,duration_group,first_obs_ordinal,intensity_class
967437,589422,40246,39b1cd89b75884a43893d6c11eb63d22,1306.635863,0.009292,overlap,1,2.735662,2.735662,2.735662,...,0.011094,0.017884,0.002233,0.0,0.059891,0.043693,1.317925,0-1 days,737680,<500
967438,589423,40246,39b1cd89b75884a43893d6c11eb63d22,81628.086905,0.580466,overlap,1,2.735662,2.735662,2.735662,...,0.024935,0.045471,0.000504,0.0,0.207131,0.132445,1.317925,0-1 days,737680,<500
967439,590825,40246,39b1cd89b75884a43893d6c11eb63d22,32481.048292,0.230976,overlap,3,13.322298,5.565268,2.735662,...,0.051346,0.102273,0.003231,0.0,0.392583,0.338587,2.661818,3+ days,737677,<500
967440,590826,40246,39b1cd89b75884a43893d6c11eb63d22,137290.471011,0.976288,overlap,3,13.322298,5.565268,2.735662,...,0.084058,0.168141,0.018623,0.0,1.196261,0.401083,2.661818,3+ days,737677,<500
967441,590827,40246,39b1cd89b75884a43893d6c11eb63d22,53775.759075,0.382405,overlap,1,5.565268,5.565268,5.565268,...,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,1.881793,0-1 days,737677,<500


In [132]:
# 6. Save outputs
safe_fire_name = FIRE_NAME.replace(" ", "_")
gpkg_path = OUTPUT_DIR / f"{safe_fire_name}_viirs_singlefire_metrics.gpkg"
summary_path = OUTPUT_DIR / f"{safe_fire_name}_summary_stats.csv"

viirs_singlefire.to_file(gpkg_path, driver="GPKG")
summary.to_csv(summary_path, index=False)

print("✅ Wrote outputs:")
print(f"  - {gpkg_path}")
print(f"  - {summary_path}")


✅ Wrote outputs:
  - /Users/dalo2903/Library/CloudStorage/OneDrive-UCB-O365/Documents/dev/intensity_x_severity/single_fire_ouputs/Holiday_Farm_viirs_singlefire_metrics.gpkg
  - /Users/dalo2903/Library/CloudStorage/OneDrive-UCB-O365/Documents/dev/intensity_x_severity/single_fire_ouputs/Holiday_Farm_summary_stats.csv


In [ ]:
# 7. Create figures (2x2 layout)
fig, axes = plt.subplots(2, 2, figsize=(14, 12), constrained_layout=True)

# --- Panel 1: log_cfrp ---
viirs_singlefire.plot(
    column="log_cfrp",
    ax=axes[0, 0],
    cmap="viridis",
    legend=True,
    linewidth=0
)
axes[0, 0].set_title("Intensity (cFRP)")
axes[0, 0].set_axis_off()

# --- Panel 2: cbi_mean ---
viirs_singlefire.plot(
    column="cbi_mean",
    ax=axes[0, 1],
    cmap="magma",
    legend=True,
    linewidth=0,
    vmin=0,
    vmax=3
)
axes[0, 1].set_title("Burn Severity (CBI)")
axes[0, 1].set_axis_off()

# --- Panel 3: duration_group ---
viirs_singlefire.plot(
    column="duration_group",
    ax=axes[1, 0],
    cmap="plasma",
    legend=True,
    linewidth=0
)
axes[1, 0].set_title("Persistence (duration)")
axes[1, 0].set_axis_off()

# --- Panel 4: Fire progression map ---
lower = viirs_singlefire["first_obs_ordinal"].quantile(0.01)
upper = viirs_singlefire["first_obs_ordinal"].quantile(0.99)
viirs_singlefire["first_obs_clipped"] = viirs_singlefire["first_obs_ordinal"].clip(lower, upper)



# Plot
ax = axes[1, 1]
viirs_singlefire.plot(
    column="first_obs_clipped",
    ax=ax,
    cmap="rainbow",
    #legend=True,
    linewidth=0
)

# Fix colorbar labels
sm = plt.cm.ScalarMappable(
    cmap="rainbow",
    norm=plt.Normalize(
        vmin=viirs_singlefire["first_obs_clipped"].min(),
        vmax=viirs_singlefire["first_obs_clipped"].max()
    )
)
sm._A = []  # needed for colorbar

cbar = fig.colorbar(sm, ax=ax)
tick_locs = np.linspace(sm.norm.vmin, sm.norm.vmax, 5)
tick_labels = [pd.Timestamp.fromordinal(int(x)).strftime("%Y-%m-%d") for x in tick_locs]
cbar.set_ticks(tick_locs)
cbar.set_ticklabels(tick_labels)

axes[1, 1].set_title("Progression (first observation date)")
axes[1, 1].set_axis_off()

# Save figure
maps_path = OUTPUT_DIR / f"{safe_fire_name}_maps_cfrp_cbi_persistence_progression.png"
fig.savefig(maps_path, dpi=200)
plt.show()
plt.close(fig)

In [ ]:
# --- Create 1x2 scatter subplots ---
fig, axes = plt.subplots(1, 2, figsize=(14, 6), constrained_layout=True)

# --- Left: hue by duration_group ---
sns.scatterplot(
    data=viirs_singlefire,
    x="log_cfrp",
    y="cbi_mean",
    hue="duration_group",
    alpha=0.6,
    ax=axes[0]
)
axes[0].set_title("Intensity vs severity by duration")
axes[0].set_xlabel("log_cfrp")
axes[0].set_ylabel("cbi_mean")

# --- Right: hue by first_obs_clipped (fire progression) ---
scatter = sns.scatterplot(
    data=viirs_singlefire,
    x="log_cfrp",
    y="cbi_mean",
    hue="first_obs_clipped",
    palette="rainbow",
    alpha=0.6,
    ax=axes[1],
    legend=False,
)
axes[1].set_title("Intensity vs severity by burn date")
axes[1].set_xlabel("log_cfrp")
axes[1].set_ylabel("cbi_mean")

# Fix colorbar to show dates
sm = plt.cm.ScalarMappable(
    cmap="rainbow",
    norm=plt.Normalize(
        vmin=viirs_singlefire["first_obs_clipped"].min(),
        vmax=viirs_singlefire["first_obs_clipped"].max()
    )
)
sm._A = []
cbar = fig.colorbar(sm, ax=axes[1])
# Convert a few ticks back to dates
tick_locs = np.linspace(sm.norm.vmin, sm.norm.vmax, 5)
tick_labels = [pd.Timestamp.fromordinal(int(x)).strftime("%Y-%m-%d") for x in tick_locs]
cbar.set_ticks(tick_locs)
cbar.set_ticklabels(tick_labels)
cbar.set_label("First Observation Date")

# --- Save figure ---
scatter_path = OUTPUT_DIR / f"{safe_fire_name}_scatter_cfrp_cbimean_dual.png"
fig.savefig(scatter_path, dpi=200)
plt.show()
plt.close(fig)

In [ ]:
gdf_clean = viirs_singlefire.dropna(subset=["cbi_mean", "frp_norm_csum", "duration_group", "intensity_class"])
fig, ax = plt.subplots(figsize=(9, 6))
sns.boxplot(
    data=gdf_clean,
    x="intensity_class",
    y="cbi_mean",
    hue="duration_group",
    palette="viridis",
    ax=ax,
)
ax.set_ylim(0, 3)
ax.set_xlabel("cFRP intensity class")
ax.set_ylabel("cbi_mean")
ax.set_title("CBI by cFRP class grouped by duration")
boxplot_path = OUTPUT_DIR / f"{safe_fire_name}_boxplot_cfrp_cbi_duration.png"
fig.savefig(boxplot_path, dpi=200)
plt.show()
plt.close(fig)